# Principia V1.3 Tutorial

This notebook demonstrates the standard real-LLM Principia V1.3 workflow.

The core stages are intentionally separate:

1. **Research**: retrieve real works from public metadata sources.
2. **Information extraction**: extract structured features from retrieved works with a real LLM.
3. **Evidence selection**: choose which extracted ideas, principles, takeaways, baselines, benchmarks, and result facts should feed generation.
4. **Generate idea**: synthesize one evidence-grounded idea with a real LLM.
5. **Compare**: check the generated idea against prior ideas extracted from the retrieved works.


## Installation And Kernel Setup

Install Principia before running this notebook:

```bash
pip install principia-ai ipykernel
```

For local source development from the package repository:

```bash
pip install -e . ipykernel
```

For VS Code, create and select a dedicated notebook kernel:

```bash
python -m ipykernel install --user --name principia-v13-python --display-name "Python 3.12 (Principia V1.3)"
```

Then open this notebook and select `Python 3.12 (Principia V1.3)`. If VS Code does not show the named kernel, select the Python interpreter from the virtual environment where `principia-ai` and `ipykernel` were installed.

This tutorial is designed for real LLM execution only.


## 1. Setup

Set `SILICONFLOW_API_KEY` in your shell before launching the notebook, or replace `YOUR_SILICONFLOW_API_KEY` in the setup cell for a local run.

The same `WORKSPACE_DIR` is used throughout the notebook so search, extraction, resume, and export all read the same project state.


In [2]:
import os

import principia as pc
from IPython.display import Markdown, display

API_key = os.environ.get("SILICONFLOW_API_KEY", "YOUR_SILICONFLOW_API_KEY")
WORKSPACE_DIR = "principia_astro"

EXTRACT_MODEL = "siliconflow:Qwen/Qwen3.6-27B"
IDEA_MODEL = "siliconflow:Deepseek/DeepSeek-V4-Pro"

goal = "Develop theory-driven, evidence-grounded hypotheses to explain possible electromagnetic emission associated with S251112cm, a sub-solar-mass compact-object merger candidate. Starting from reported optical transient candidates and existing follow-up constraints, synthesize principles from kilonovae, super-kilonovae, kilonovae-within-supernovae, AGN-disk merger flares, and ordinary supernova contaminants, and infer which physical scenarios can consistently explain the observed temporal, spatial, photometric, spectroscopic, and host-galaxy evidence."

pc.__version__


'1.3.2'

Expected output: `1.3.2`.

## 2. Create Workspace

The workspace stores user-facing outputs in visible files and internal state in `.principia/`:

```text
principia_astro/
  README.md
  principia_outputs/
    latest/
      idea.md
      result.json
      works.json
    exports/
      <idea_id>/
        idea.md
        result.json
        works.json
  .principia/
    principia.sqlite
    artifacts/
      source_json/
      exports/
      pdfs/
      cache/
```


In [3]:


ws = pc.Workspace(
    WORKSPACE_DIR,
    llm_config=pc.siliconflow_config(API_key, timeout=900, max_retries=4),
)

ws.path


PosixPath('/home/supernova/Desktop/AI_projects/Principia/Principia-v1.3/examples/principia_astro')

## 3. Research: Search Real Works

This cell retrieves 50 real works from public metadata sources and stores them in SQLite.

Principia now uses the shared semantic retriever over arXiv, OpenAlex, Crossref, and Semantic Scholar. It plans multiple academic search queries from the goal, deduplicates candidates by DOI/arXiv/OpenAlex/Semantic Scholar IDs and normalized title, and uses LLM semantic reranking when a real LLM is configured. Without an LLM it falls back to deterministic semantic-lite ranking.

For non-AI goals, AI/LLM/MAS query expansion is not triggered unless the goal explicitly asks for AI, LLMs, agents, or MAS.

The progress display is provided by Principia. It includes elapsed time and ETA.


In [4]:

search_progress = pc.notebook_progress("Research search")

works = ws.research.search(
    goal,
    target_count=50,
    callback=search_progress,
)

len(works), works[0].title


### Research search

**Operation:** `research.search`  
**Stage:** `complete`  
**Progress:** `####################` 100%  
**Elapsed:** 2m 47s  
**ETA:** 0s  
**Message:** Complete.  
**Counts:** target_count=50, query_count=8, entities=['S251112cm', 'sub-solar-mass compact objects', 'primordial black holes', 'kilonovae', 'super-kilonovae', 'active galactic nucleus (AGN) disks', 'core-collapse supernovae', 'Type Ia supernovae'], sources=['arxiv', 'openalex', 'crossref', 'semantic_scholar'], raw_candidates=76, selected_candidates=50, works=50

(50,
 'Search For a Counterpart to the Subsolar Mass Gravitational Wave Candidate S251112cm')

## 4. Review Retrieved Works

A `WorkItem` stores metadata such as `id`, `title`, `authors`, `abstract`, `year`, `venue`, `url`, `doi`, `arxiv_id`, and `openalex_id`.

The table below shows the top 12 retrieved works. Re-run the research cell after changing the goal or retriever code so the displayed works reflect the current search implementation.


In [5]:
work_rows = [
    [i + 1, w.title, pc.work_review_status(w), w.year or "", w.venue or w.source, w.url]
    for i, w in enumerate(works)
]
display(Markdown(pc.markdown_table(["#", "Title", "Review status", "Year", "Venue/Source", "URL"], work_rows)))


| # | Title | Review status | Year | Venue/Source | URL |
| --- | --- | --- | --- | --- | --- |
| 1 | Search For a Counterpart to the Subsolar Mass Gravitational Wave Candidate S251112cm | preprint | 2026 | arXiv (Cornell University) | https://doi.org/10.48550/arxiv.2603.17009 |
| 2 | Electromagnetic Follow-up of the Sub-Solar Mass Gravitational Wave Candidate S251112cm: Kilonova Constraints and a Coincident IIb Supernova | preprint | 2026 | arXiv (Cornell University) | https://arxiv.org/abs/2605.10940 |
| 3 | The Electromagnetic Counterpart of the Binary Neutron Star Merger LIGO/Virgo GW170817. I. Discovery of the Optical Counterpart Using the Dark Energy Camera | unknown | 2017 | The Astrophysical Journal Letters | https://doi.org/10.3847/2041-8213/aa9059 |
| 4 | The Emergence of a Lanthanide-rich Kilonova Following the Merger of Two Neutron Stars | unknown | 2017 | The Astrophysical Journal Letters | https://doi.org/10.3847/2041-8213/aa90b6 |
| 5 | Possible role of magnetic reconnection in the electromagnetic counterpart of binary black hole merger | unknown | 2018 | Journal of Cosmology and Astroparticle Physics | https://doi.org/10.1088/1475-7516/2018/04/054 |
| 6 | Kilonova Light Curves from the Disk Wind Outflows of Compact Object Mergers | preprint | 2014 | arXiv | http://arxiv.org/abs/1411.3726v1 |
| 7 | Implications for Primordial Black Hole Dark Matter from a Single Subsolar Mass Gravitational-wave Detection in LVK O1–O4 | preprint | 2026 | The Astrophysical Journal | https://doi.org/10.3847/1538-4357/ae48f9 |
| 8 | Candidate Electromagnetic Counterpart to the Binary Black Hole Merger Gravitational-Wave Event S190521g | preprint | 2020 | Physical Review Letters | https://doi.org/10.1103/physrevlett.124.251102 |
| 9 | Swope Supernova Survey 2017a (SSS17a), the optical counterpart to a gravitational wave source | unknown | 2017 | Science | https://doi.org/10.1126/science.aap9811 |
| 10 | Filling the disk hollow following binary black hole merger: The transient accretion afterglow | unknown | 2010 | Physical Review D | https://doi.org/10.1103/physrevd.81.024019 |
| 11 | Optical emission from a kilonova following a gravitational-wave-detected neutron-star merger | unknown | 2017 | Nature | https://doi.org/10.1038/nature24291 |
| 12 | A kilonova as the electromagnetic counterpart to a gravitational-wave source | unknown | 2017 | Nature | https://doi.org/10.1038/nature24303 |
| 13 | Primordial Black Hole interpretation of the sub-solar merger event S251112cm | preprint | 2026 | arXiv (Cornell University) | http://arxiv.org/abs/2603.25795 |
| 14 | Unveiling the Kilonova of GRB 211211A: A Unique Long-Duration Gamma-Ray Burst from a Compact Object Merger | unknown | 2024 | International Research Journal on Advanced Engineering and Management (IRJAEM) | https://doi.org/10.47392/irjaem.2024.0412 |
| 15 | A comparison between SALT/SAAO observations and kilonova models for AT 2017gfo: the first electromagnetic counterpart of a gravitational wave transient - GW170817 | preprint | 2017 | arXiv | http://arxiv.org/abs/1710.05855v1 |
| 16 | WHAT IS THE MOST PROMISING ELECTROMAGNETIC COUNTERPART OF A NEUTRON STAR BINARY MERGER? | unknown | 2012 | The Astrophysical Journal | https://doi.org/10.1088/0004-637x/746/1/48 |
| 17 | Searching for electromagnetic emission in an AGN from the gravitational wave binary black hole merger candidate S230922g | unknown | 2024 | Physical Review D | https://www.semanticscholar.org/paper/6d337d6992b23e714d864648cab58f0baf256055 |
| 18 | Possible Flare from Black Hole Merger | unknown | 2020 | Physics | https://doi.org/10.1103/physics.13.101 |
| 19 | Long-Term Optical Follow Up of S231206cc: Multi-Model Constraints on BBH Merger Emission in AGN Disks | preprint | 2025 | arXiv | http://arxiv.org/abs/2506.02224v1 |
| 20 | Tracing the light: Identification for the optical counterpart candidates of binary black-holes during O3 | preprint | 2025 | arXiv | http://arxiv.org/abs/2507.02475v1 |
| 21 | Binary Black Hole Formation in AGN disk using Athena++ [Slides] | unknown | 2023 | Office of Scientific and Technical Information (OSTI) | https://doi.org/10.2172/1989144 |
| 22 | Accretion-modified Stars in Accretion Disks of Active Galactic Nuclei: Slowly Transient Appearance | preprint | 2021 | The Astrophysical Journal Letters | https://doi.org/10.3847/2041-8213/abee81 |
| 23 | Searching for Binary Black Hole Merger Emission in AGN Disks: Optical and Spectroscopic Follow-up of S240413p | preprint | 2026 | Semantic Scholar | https://www.semanticscholar.org/paper/fe66713359437dd5e46a190e65bb0d26966e3023 |
| 24 | Origin of the heavy elements in binary neutron-star mergers from a gravitational-wave event | unknown | 2017 | Nature | https://doi.org/10.1038/nature24453 |
| 25 | Ram-pressure Stripping of a Kicked Hill Sphere: Prompt Electromagnetic Emission from the Merger of Stellar Mass Black Holes in an AGN Accretion Disk | unknown | 2019 | Astrophysical Journal | https://www.semanticscholar.org/paper/013801ece3d672ae8a19d2ec9e5b00df8ba608ec |
| 26 | Detectable Environmental Effects in GW190521-like Black-Hole Binaries with LISA | unknown | 2021 | Physical Review Letters | https://doi.org/10.1103/physrevlett.126.101105 |
| 27 | Slim-disk modeling reveals an accreting intermediate-mass black hole in the luminous fast blue optical transient AT2018cow | unknown | 2024 | Astronomy &amp; Astrophysics | https://www.semanticscholar.org/paper/f028cf98aabd48aa24211c7c72165e14b04a1242 |
| 28 | Understanding extreme quasar optical variability with CRTS: I. Major AGN flares | preprint | 2017 | arXiv | http://arxiv.org/abs/1706.03079v1 |
| 29 | A dust-enshrouded tidal disruption event with a resolved radio jet in a galaxy merger | unknown | 2018 | Science | https://doi.org/10.1126/science.aao4669 |
| 30 | A Deep CFHT Optical Search for a Counterpart to the Possible Neutron Star -- Black Hole Merger GW190814 | preprint | 2020 | arXiv | http://arxiv.org/abs/2003.09437v3 |
| 31 | Radioactive Heating and Late Time Kilonova Light Curves | preprint | 2018 | arXiv | http://arxiv.org/abs/1807.03319v1 |
| 32 | A Challenge to Identify an Optical Counterpart of the Gravitational Wave Event GW151226 with Hyper Suprime-Cam | preprint | 2017 | arXiv | http://arxiv.org/abs/1710.00127v1 |
| 33 | On the impact of compact binary merger ejecta opacity on Kilonova transient signals | unknown | 2023 | EPJ Web of Conferences | https://doi.org/10.1051/epjconf/202327502012 |
| 34 | The optical electromagnetic counterpart of the gravitational wave event GW170817 | unknown | 2019 | Nuclear and Particle Physics Proceedings | https://doi.org/10.1016/j.nuclphysbps.2019.07.006 |
| 35 | Searching for Electromagnetic Counterpart Candidates to GW231123 | unknown | 2025 | Semantic Scholar | https://www.semanticscholar.org/paper/464ae245a166ed2ffe5023eaca1f45dc7626d2dd |
| 36 | A Light in the Dark: Searching for Electromagnetic Counterparts to Black Hole–Black Hole Mergers in LIGO/Virgo O3 with the Zwicky Transient Facility | unknown | 2023 | The Astrophysical Journal | https://doi.org/10.3847/1538-4357/aca480 |
| 37 | Insights from GRBs for optical follow-up of gravitational wave counterparts | preprint | 2026 | arXiv | http://arxiv.org/abs/2604.01485v1 |
| 38 | Identifying potential binary neutron star merger events from the Fermi GBM Gamma-Ray Burst Catalog | preprint | 2025 | arXiv | http://arxiv.org/abs/2507.18258v1 |
| 39 | Gravitational wave optical counterpart searching based on GRAWITA and DLT40 project during LIGO O2 run | unknown | 2017 | Proceedings of the International Astronomical Union | https://doi.org/10.1017/s1743921318000157 |
| 40 | A Comprehensive Study of Detectability and Contamination in Deep Rapid Optical Searches for Gravitational Wave Counterparts | preprint | 2015 | arXiv | http://arxiv.org/abs/1503.07869v1 |
| 41 | Search for Sub-Solar Mass Binaries in the First Part of LIGO's Fourth Observing Run | unknown | 2026 | arXiv (Cornell University) | http://arxiv.org/abs/2602.12115 |
| 42 | Constraining the Fraction of LIGO/Virgo/KAGRA Binary Black Hole Merger Events Associated with Active Galactic Nucleus Flares | preprint | 2026 | arXiv | http://arxiv.org/abs/2601.08286v2 |
| 43 | Probing (sub-)solar-mass black holes and superspinars with current and next-generation gravitational-wave observatories | unknown | 2026 | arXiv (Cornell University) | https://arxiv.org/abs/2605.18428 |
| 44 | The Discovery of the Electromagnetic Counterpart of GW170817: Kilonova AT 2017gfo/DLT17ck | unknown | 2017 | The Astrophysical Journal Letters | https://doi.org/10.3847/2041-8213/aa8edf |
| 45 | Search for Sub-Solar Mass Binaries with Einstein Telescope and Cosmic Explorer | unknown | 2022 | Entropy | https://doi.org/10.3390/e24020262 |
| 46 | Properties and Astrophysical Implications of the 150 M<sub>⊙</sub> Binary Black Hole Merger GW190521 | unknown | 2020 | The Astrophysical Journal Letters | https://doi.org/10.3847/2041-8213/aba493 |
| 47 | Measuring the Obscuring Column of a Disk Megamaser AGN in a Nearby Merger | unknown | 2019 | The Astrophysical Journal | https://doi.org/10.3847/1538-4357/ab3214 |
| 48 | Superheavy Elements in Kilonovae | preprint | 2023 | arXiv | http://arxiv.org/abs/2304.02125v2 |
| 49 | AGNs on the Move: A Search for Off-nuclear AGNs from Recoiling Supermassive Black Holes and Ongoing Galaxy Mergers with the Zwicky Transient Facility | unknown | 2021 | The Astrophysical Journal | https://doi.org/10.3847/1538-4357/abf246 |
| 50 | Prospects for kilonova signals in the gravitational-wave era | unknown | 2021 | Astronomy &amp; Astrophysics | https://doi.org/10.1051/0004-6361/202140689 |

## 5. Information Extraction

This cell extracts structured research features from the top-ranked works using DeepSeek.

For a tutorial, extracting the top 6 works keeps runtime and provider timeout risk manageable while preserving the full 50-work research list. Increase `EXTRACT_COUNT` and `max_chars` after the first run succeeds.

The progress display includes ETA and per-item counts. During a provider call, Principia keeps the display alive with heartbeat updates rather than freezing on a fixed percentage.


In [6]:

EXTRACT_COUNT = min(20, len(works))
extract_progress = pc.notebook_progress("Information extraction")

features = ws.research.extract(
    works[:EXTRACT_COUNT],
    model=EXTRACT_MODEL,
    overwrite=False,
    max_chars=12_000,
    continue_on_error=True,
    callback=extract_progress,
)

features.counts(), features.model


### Information extraction

**Operation:** `research.extract`  
**Stage:** `complete`  
**Progress:** `####################` 100%  
**Elapsed:** 12m 48s  
**ETA:** 0s  
**Message:** Complete.  
**Counts:** work_id=W-7595EB1FCE77, current=20, total=20, extracted=20, skipped=4, llm_wait_seconds=50

({'works': 20,
  'ideas': 37,
  'principles': 26,
  'takeaways': 34,
  'baselines': 10,
  'benchmarks': 11,
  'result_facts': 74},
 'siliconflow:Qwen/Qwen3.6-27B')

## 6. Review Extracted Features

`ExtractedFeatures` is a batch object. Each item is a `WorkFeatures` record with these feature buckets:

- `ideas`: prior or existed ideas extracted from the work.
- `principles`: reusable principles or mechanisms.
- `takeaways`: actionable lessons or result messages.
- `baselines`: comparison methods.
- `benchmarks`: datasets, tasks, or evaluation settings.
- `result_facts`: grounded factual results.

Each extracted record has an `id`. You can use those IDs in the next step to select specific evidence for idea generation.


In [7]:

display(Markdown(pc.feature_summary_markdown(features, limit=8)))

| Work ID | Work title | Existed idea | Principle | Takeaway |
| --- | --- | --- | --- | --- |
| W-D73E3885317D | Search For a Counterpart to the Subsolar Mass Gravitational Wave Candid... | A systematic framework to vet and score candidate electromagnetic counterparts to subsolar mass (SSM) gravitational wav... | SSM mergers likely do not involve the supersolar neutron stars or black holes typically invoked to explain kilonovae. | Despite extensive follow-up, no likely electromagnetic counterpart was identified for the S251112cm event. |
| W-8565AD9F232C | Electromagnetic Follow-up of the Sub-Solar Mass Gravitational Wave Cand... | A theoretical model proposing that sub-solar mass neutron star mergers can be generated from stellar core-collapse via... | For the superkilonova model to be valid, the supernova explosion must precede the gravitational wave merger by a short... | No kilonova counterpart was found for S251112cm, allowing researchers to rule out a significant portion of kilonova mod... |
| W-DD861586E01A | The Electromagnetic Counterpart of the Binary Neutron Star Merger LIGO/... | The identification of the optical transient associated with the binary neutron star merger GW170817 using the Dark Ener... | Gravitational wave detections can be validated and localized through the identification of electromagnetic counterparts... | The Dark Energy Camera is a powerful tool for identifying optical counterparts of gravitational-wave sources due to its... |
| W-179FA0EF06AE | The Emergence of a Lanthanide-rich Kilonova Following the Merger of Two... | The transient light from the neutron-star merger counterpart (AT2017gfo) is powered by the radioactive decay of massive... | Neutron-star mergers are confirmed to be major, if not dominant, sites for rapid neutron capture nucleosynthesis in the... | The observation of AT2017gfo confirms that neutron-star mergers produce kilonovae/macronovae. |
| W-7F74BE41DDBD | Possible role of magnetic reconnection in the electromagnetic counterpa... |  |  |  |
| W-5601C1DD2ACB | Kilonova Light Curves from the Disk Wind Outflows of Compact Object Mer... | Kilonova transients from accretion disk winds exhibit two distinct components based on electron fraction (Ye) and resul... | The electron fraction (Ye) of the ejecta dictates the extent of r-process nucleosynthesis, which in turn determines the... | Disk wind models predict optical emission, which improves the chances of detecting electromagnetic counterparts to grav... |
| W-347BC8098122 | Implications for Primordial Black Hole Dark Matter from a Single Subsol... | Primordial Black Holes (PBHs) formed during the Quantum Chromodynamics (QCD) epoch can account for subsolar mass gravit... | Constraints derived assuming monochromatic mass distributions are unreliable for extended mass functions because they d... | The observed merger rate of the candidate event S251112cm is compatible with a model of PBHs formed at the QCD epoch. |
| W-E573123E4055 | Candidate Electromagnetic Counterpart to the Binary Black Hole Merger G... | A binary black hole (BBH) merger occurring within the accretion disk of an active galactic nucleus (AGN) can produce a... | The lack of color evolution in the optical flare indicates a constant temperature shock rather than a cooling ejecta, w... | If the flare is caused by a kicked black hole moving through an AGN disk, a repeat flare is expected when the black hol... |

## Optional: Start A New Notebook From Existing Features

If you already completed research and information extraction in this workspace, a new notebook can resume from persisted SQLite records.

Use the same workspace folder and call `ws.load_features()`. This does not rerun public search or LLM extraction.


In [8]:
resumed_ws = pc.Workspace(
    WORKSPACE_DIR,
    llm_config=pc.siliconflow_config(API_key, timeout=900, max_retries=4),
)

resumed_features = resumed_ws.load_features()
resumed_features.counts()


{'works': 21,
 'ideas': 38,
 'principles': 27,
 'takeaways': 36,
 'baselines': 10,
 'benchmarks': 11,
 'result_facts': 76}

## 7. Select Evidence For Idea Generation

By default, `pc.select_evidence(features)` selects all extracted feature buckets.

You can narrow the generation input with:

- `kinds=["ideas", "principles", "takeaways"]`
- `work_ids=[...]`
- `feature_ids=[...]`
- `limit_per_kind=...`

The result is an `EvidencePacket`, which is the direct input to `ws.ideas.generate(...)`.


In [9]:
selected_evidence = pc.select_evidence(
    features,
    kinds=["ideas", "principles", "takeaways", "baselines", "benchmarks", "result_facts"],
)

selected_evidence.counts()


{'works': 16,
 'ideas': 37,
 'principles': 26,
 'takeaways': 34,
 'baselines': 10,
 'benchmarks': 11,
 'result_facts': 74}

## 8. Generate Idea

This cell generates one evidence-grounded idea from `selected_evidence` using Qwen.

The generated `Idea` includes thesis, novelty claim, mechanistic design, methodological details, method variants, validation protocol, metrics, risks, assumptions, source evidence, lineage or trace metadata, and generation metadata.


In [10]:

generate_progress = pc.notebook_progress("Idea generation")

idea = ws.ideas.generate(
    selected_evidence,
    user_note=goal,
    mode="calculus",
    model=IDEA_MODEL,
    callback=generate_progress,
)

idea.title


### Idea generation

**Operation:** `ideas.generate.calculus`  
**Stage:** `error`  
**Progress:** `#######.............` 37%  
**Elapsed:** 14s  
**ETA:** calculating  
**Message:** LLM call failed: Provider returned HTTP 400: {"code":20012,"message":"Model does not exist. Please check it carefully.","data":null}  
**Counts:** evidence_items=16, llm_wait_seconds=12

RuntimeError: LLM call failed: Provider returned HTTP 400: {"code":20012,"message":"Model does not exist. Please check it carefully.","data":null}

## 9. Review Generated Idea

`pc.idea_markdown(...)` renders the full idea card, including Methodological Details and LaTeX equations when present.


In [ ]:

display(Markdown(pc.idea_markdown(idea)))


## Eccentricity-Driven Spectral Signatures in Collapsar-Disk Subsolar Mergers

**ID:** `Eccentricity_Driven_Spectral_Signatures_In_Collapsar_Disk_Subsolar_Mergers`  
**Mode:** `calculus`  
**Model:** `siliconflow:Qwen/Qwen3.5-397B-A17B`

**Thesis:** If the sub-solar mass candidate S251112cm originated from hierarchical fragmentation within a collapsar accretion disk, the surviving orbital eccentricity ($e \sim 0.1$) at merger will induce distinct, non-thermal spectral hardening and time-variable polarization in the post-merger electromagnetic counterpart, distinguishing it from circularized primordial black hole mergers or standard kilonovae.

### Novelty Claim
- While eccentricity is known as a dynamical signature of hierarchical formation in collapsar disks, this proposal uniquely links the specific surviving eccentricity magnitude ($e \approx 0.1$) predicted for sub-solar fragments to observable non-thermal spectral features and polarization variability in the electromagnetic domain, providing a multi-messenger discriminant against the Primordial Black Hole (PBH) hypothesis which predicts circular orbits.

### Mechanistic Design
- {'summary': 'The mechanism posits that sub-solar fragments formed via disk fragmentation undergo repeated captures before merging. The short system lifetime prevents full circularization, leaving residual eccentricity. This eccentricity drives periodic shocks and variable accretion rates in the post-merger remnant disk, generating non-thermal emission distinct from the thermal decay of standard kilonovae or the absence of emission in vacuum PBH mergers.', 'symbols': {'e': 'Orbital eccentricity at merger', 't_visc': 'Viscous timescale of the post-merger disk', 'P_orb': 'Orbital period of the eccentric remnant', 'L_nt': 'Non-thermal luminosity from shock heating', 'f_PBH': 'Fraction of dark matter in primordial black holes', 'R_hier': 'Merger rate for hierarchical collapsar channel'}, 'equations': [{'name': 'Eccentricity Survival Condition', 'latex': '$$e_{final} \\approx e_{initial} \\left( \\frac{t_{merge}}{t_{circ}} \\right)^{-1/2}$$', 'explanation': 'Estimates the final eccentricity $e_{final}$ given the initial eccentricity $e_{initial} \\sim 0.6$, where $t_{merge}$ is the time to merger and $t_{circ}$ is the circularization timescale. For sub-solar fragments in dense disks, $t_{merge} < t_{circ}$, allowing $e_{final} \\sim 0.1$ to survive.'}, {'name': 'Modulated Shock Luminosity', 'latex': '$$L_{nt}(t) \\propto \\dot{M}(t)^2 \\left( 1 + e \\cos(\\theta(t)) \\right)^4$$', 'explanation': 'Describes the non-thermal luminosity $L_{nt}$ generated by shocks in the accretion flow, modulated by the orbital phase $\\theta(t)$ and eccentricity $e$. The strong dependence on $(1+e \\cos \\theta)$ creates observable periodic flaring absent in circular mergers.'}, {'name': 'Hierarchical Rate Constraint', 'latex': '$$R_{hier} \\propto f_{disk} \\times \\Sigma_{frag} \\times P_{capture}$$', 'explanation': 'Defines the merger rate $R_{hier}$ for the hierarchical channel as a function of the fraction of collapsars with fragmenting disks $f_{disk}$, the surface density of fragments $\\Sigma_{frag}$, and the capture probability $P_{capture}$, independent of the PBH abundance $f_{PBH}$.'}], 'workflow': ['Ingest GW parameters for S251112cm', 'Simulate eccentric disk hydrodynamics', 'Generate synthetic spectral energy distributions', 'Cross-match with ZTF/DECam non-detections', 'Compute Bayesian odds ratio for eccentric model', 'Derive polarization variability predictions'], 'reliability_checks': ['Verify that simulated $e_{final}$ matches the theoretical bound of $\\sim 0.1$ from numerical relativity.', 'Ensure synthetic light curves do not exceed the flux limits set by DECam and FTW non-detections.', 'Confirm that the predicted non-thermal spectrum differs significantly from the thermal blackbody of a standard kilonova.']}

### Methodological Details

We will employ relativistic hydrodynamics simulations to model the post-merger evolution of an eccentric sub-solar binary within a collapsar disk environment. Using the surviving eccentricity $e \approx 0.1$ as a fixed parameter derived from prior hierarchical merger studies, we will calculate the resulting time-dependent accretion rate and shock heating. These physical outputs will be fed into radiative transfer codes to generate synthetic light curves and spectra, which will then be compared against the existing non-detection constraints from DECam, FTW, and ZTF for S251112cm.

#### Symbols
- 

#### Equations
- **Eccentricity Survival Condition:** $$e_{final} \approx e_{initial} \left( \frac{t_{merge}}{t_{circ}} \right)^{-1/2}$$ — Estimates the final eccentricity $e_{final}$ given the initial eccentricity $e_{initial} \sim 0.6$, where $t_{merge}$ is the time to merger and $t_{circ}$ is the circularization timescale. For sub-solar fragments in dense disks, $t_{merge} < t_{circ}$, allowing $e_{final} \sim 0.1$ to survive.
- **Modulated Shock Luminosity:** $$L_{nt}(t) \propto \dot{M}(t)^2 \left( 1 + e \cos(\theta(t)) \right)^4$$ — Describes the non-thermal luminosity $L_{nt}$ generated by shocks in the accretion flow, modulated by the orbital phase $\theta(t)$ and eccentricity $e$. The strong dependence on $(1+e \cos \theta)$ creates observable periodic flaring absent in circular mergers.
- **Hierarchical Rate Constraint:** $$R_{hier} \propto f_{disk} \times \Sigma_{frag} \times P_{capture}$$ — Defines the merger rate $R_{hier}$ for the hierarchical channel as a function of the fraction of collapsars with fragmenting disks $f_{disk}$, the surface density of fragments $\Sigma_{frag}$, and the capture probability $P_{capture}$, independent of the PBH abundance $f_{PBH}$.

#### Workflow
1. **Step:** Ingest GW parameters for S251112cm
2. **Step:** Simulate eccentric disk hydrodynamics
3. **Step:** Generate synthetic spectral energy distributions
4. **Step:** Cross-match with ZTF/DECam non-detections
5. **Step:** Compute Bayesian odds ratio for eccentric model
6. **Step:** Derive polarization variability predictions

#### Reliability Checks
- **Step:** Verify that simulated $e_{final}$ matches the theoretical bound of $\sim 0.1$ from numerical relativity.
- **Step:** Ensure synthetic light curves do not exceed the flux limits set by DECam and FTW non-detections.
- **Step:** Confirm that the predicted non-thermal spectrum differs significantly from the thermal blackbody of a standard kilonova.

### Method Variants
- Variant A: Assume a 'relaxed' microlensing constraint on PBH abundance to test if the eccentric model is required even if PBHs are abundant.
- Variant B: Model the counterpart as a 'kilonova-within-supernova' where the eccentric merger occurs inside the expanding ejecta of the progenitor Type IIb supernova (SN 2025adtq analog).
- Variant C: Treat the eccentricity as a free parameter to derive an upper limit on $e$ based on the smoothness of the existing ZTF light curve limits.

### Derived Principles
- Eccentricity as a Signature of Hierarchical Formation: Surviving eccentricity in sub-solar mergers indicates a dynamic, dense-environment origin rather than isolated binary evolution or primordial origins.
- Subsolar Mass as Non-Stellar Signature: Detections below $1 M_{\odot}$ challenge standard stellar collapse, requiring either exotic astrophysical channels (collapsars) or early-Universe physics (PBHs).
- Temporal Association Constraint: Validating formation channels involving supernovae requires precise temporal coincidence ($< $ few days) between the explosion and the GW merger.

### Why It Might Work
- Standard kilonova models have already been ruled out for 42-92% of the parameter space for S251112cm, and the PBH hypothesis predicts no electromagnetic emission. The collapsar disk scenario is the only remaining channel that naturally explains both the sub-solar mass and the potential for a faint, non-thermal counterpart. The specific prediction of surviving eccentricity ($e \sim 0.1$) provides a unique 'fingerprint' that can be searched for in re-analyzed data or future events, bypassing the ambiguity of simple brightness constraints.

### Validation Protocol
- 1. Re-analyze the raw ZTF and DECam image data for S251112cm using difference imaging tuned for fast, non-thermal transients rather than standard kilonova templates. 2. Search for periodic residuals consistent with the $P_{orb}$ derived from the GW chirp mass and $e \sim 0.1$. 3. If no signal is found, establish an upper limit on the non-thermal efficiency $\eta_{nt}$ of eccentric mergers. 4. Compare the derived limits against the predicted rates for hierarchical mergers versus the PBH rate of $0.8 \text{ yr}^{-1}$.

### Relevant Baselines
- Standard Neutron Star Formation: Predicts no sub-solar mass mergers.
- Primordial Black Hole (PBH) Merger: Predicts circular orbits and no electromagnetic counterpart.
- Standard Kilonova: Predicts thermal emission already largely ruled out by DECam/FTW non-detections.

### Metrics
- Bayesian Odds Ratio ($\mathcal{O}_{ecc/pbh}$) comparing the eccentric disk model to the PBH null hypothesis.
- Upper limit on non-thermal luminosity $L_{nt}$ at 95% confidence.
- Constraint on surviving eccentricity $e_{final}$.
- Consistency of the hierarchical merger rate $R_{hier}$ with the observed rate of sub-solar candidates.

### Risks
- The electromagnetic counterpart may be too faint to detect even with optimized non-thermal models, leading to inconclusive results similar to the current status of S251112cm.
- The association between S251112cm and any optical transient (like SN 2025adtq) may be a chance coincidence, undermining the physical basis for the collapsar disk model.
- Uncertainties in the detector sensitivity scaling for sub-solar masses ($M^{2.34}$) may bias the rate comparisons.

### Assumptions
- Sub-solar mass objects can form via fragmentation in collapsar disks.
- The system lifetime is short enough to preserve eccentricity $e \sim 0.1$ until merger.
- Eccentric accretion flows generate detectable non-thermal radiation distinct from thermal kilonovae.
- The GW candidate S251112cm is of astrophysical origin and not a noise artifact.

### Source Evidence
- **ideas / Superkilonova Formation Channel:** A theoretical model proposing that sub-solar mass neutron star mergers can be generated from stellar core-collapse via disk fragmentation.
- **ideas / Electromagnetic Counterpart Search for Sub-Solar Mass GWs:** A multi-facility observational campaign to identify electromagnetic counterparts (kilonovae or supernovae) for sub-solar mass gravitational wave candidates.
- **principles / Temporal Association Constraint:** For the superkilonova model to be valid, the supernova explosion must precede the gravitational wave merger by a short duration.
- **principles / Statistical Association vs. Chance Coincidence:** The significance of a spatial and temporal association between a GW event and a supernova must be evaluated against the probability of chance coincidence.
- **takeaways / Kilonova Non-Detection Constraints:** No kilonova counterpart was found for S251112cm, allowing researchers to rule out a significant portion of kilonova models.
- **takeaways / Inconclusive Evidence for Superkilonova Channel:** The discovery of SN 2025adtq provides suggestive but not conclusive evidence for the superkilonova formation channel.
- **baselines / Standard Neutron Star Formation:** Standard astrophysical models for neutron star formation do not typically predict sub-solar mass components in mergers.
- **benchmarks / GW Candidate S251112cm Parameters:** Characterization of the gravitational wave candidate used as the primary subject of the study.
- **benchmarks / Previous Candidate S250818k:** A prior sub-solar mass GW candidate used for comparative analysis.
- **result_facts / Survey Coverage:** Introduction: 'Combined, these three telescopes surveyed ~56% of the probability region with at least two filters within 48 hours after merger.'
- **result_facts / Kilonova Model Exclusion Rates:** Abstract: 'rule out 42% (ZTF), 68% (DECam), and 92% (FTW) of the KN models as possible emission from this GW candidate.'
- **result_facts / SN 2025adtq Association Metrics:** Abstract: 'spatial association odds ratio of $\log_{10}\mathcal{I} \approx 4.8$, a chance coincidence probability of ~2–9%, and an estimated explosion time ~2 days prior to S251112cm.'
- **result_facts / Joint Odds Ratio Analysis:** Abstract: 'jointly, we measure an odds ratio that favors the association hypothesis over the null, however, when conditioned on finding a coincident supernova by chance, the odds ratio disfavors association.'
- **ideas / QCD Epoch Primordial Black Hole Population:** A population of Primordial Black Holes (PBHs) formed during the quantum chromodynamics (QCD) epoch with a broad, extended mass function can account for subsolar mass gravitational-wave detections.
- **ideas / PBHs as Dark Matter Candidates:** Primordial Black Holes could constitute a significant fraction, or potentially all, of the Universe's Dark Matter (DM).
- **principles / Subsolar Mass as Non-Stellar Signature:** Compact objects with masses less than 1 solar mass ($M less 1 M_{igodot}$) are not expected to form through conventional stellar collapse mechanisms. Therefore, their detection strongly supports a non-stellar origin, s...
- **principles / Consistency with Stellar-Mass Binary Rates:** The predicted detection rate of PBH mergers must be consistent with current LVK expectations for stellar-mass binaries to remain a viable model.
- **takeaways / Compatibility of S251112cm with QCD PBH Model:** The candidate event S251112cm is statistically compatible with a population of PBHs formed at the QCD epoch.
- **takeaways / Lower Limit on PBH Abundance:** Confirmation of the subsolar mass detection places a lower limit on the fraction of Dark Matter composed of PBHs.
- **baselines / Standard Stellar Collapse Models:** Traditional astrophysical models for black hole formation via stellar evolution.
- **baselines / LVK O3b Observed Merger Rate:** The observed merger rate of compact binaries from the LIGO-Virgo-KAGRA O3b observing run.
- **benchmarks / Event S251112cm Characteristics:** The specific gravitational-wave candidate used to motivate the study.
- **result_facts / Predicted PBH Event Rate:** Predicted PBH Event Rate
- **result_facts / Lower Limit on PBH Dark Matter Fraction:** Lower Limit on PBH Dark Matter Fraction

### Lineage
```json
{
  "nodes": [
    {
      "id": "W-8565AD9F232C",
      "type": "work",
      "label": "Electromagnetic Follow-up of the Sub-Solar Mass Gravitational Wave Candidate S251112cm: Kilonova Constraints and a Coincident IIb Supernova"
    },
    {
      "id": "W-347BC8098122",
      "type": "work",
      "label": "Implications for Primordial Black Hole Dark Matter from a Single Subsolar Mass Gravitational-wave Detection in LVK O1–O4"
    },
    {
      "id": "W-D73E3885317D",
      "type": "work",
      "label": "Search For a Counterpart to the Subsolar Mass Gravitational Wave Candidate S251112cm"
    },
    {
      "id": "W-F2950CEADAAF",
      "type": "work",
      "label": "Primordial Black Hole interpretation of the sub-solar merger event S251112cm"
    },
    {
      "id": "W-DA814CF26D9A",
      "type": "work",
      "label": "Eccentricity as a Signature of Hierarchical Subsolar-mass Mergers in Collapsar Disks"
    },
    {
      "id": "W-7595EB1FCE77",
      "type": "work",
      "label": "Tracing the light: Identification for the optical counterpart candidates of binary black-holes during O3"
    },
    {
      "id": "D_EvidenceGate",
      "type": "derived_concept",
      "label": "Evidence-gated mechanism"
    }
  ],
  "edges": [
    {
      "source": "W-8565AD9F232C",
      "target": "D_EvidenceGate",
      "relation": "supports"
    },
    {
      "source": "W-347BC8098122",
      "target": "D_EvidenceGate",
      "relation": "supports"
    },
    {
      "source": "W-D73E3885317D",
      "target": "D_EvidenceGate",
      "relation": "supports"
    },
    {
      "source": "W-F2950CEADAAF",
      "target": "D_EvidenceGate",
      "relation": "supports"
    },
    {
      "source": "W-DA814CF26D9A",
      "target": "D_EvidenceGate",
      "relation": "supports"
    },
    {
      "source": "W-7595EB1FCE77",
      "target": "D_EvidenceGate",
      "relation": "supports"
    }
  ]
}
```

### Generation Metadata
```json
{
  "mode": "calculus",
  "model": "siliconflow:Qwen/Qwen3.5-397B-A17B",
  "evidence_counts": {
    "works": 6,
    "ideas": 12,
    "principles": 10,
    "takeaways": 13,
    "baselines": 5,
    "benchmarks": 6,
    "result_facts": 18
  },
  "selected_work_ids": [
    "W-8565AD9F232C",
    "W-347BC8098122",
    "W-D73E3885317D",
    "W-F2950CEADAAF",
    "W-DA814CF26D9A",
    "W-7595EB1FCE77"
  ]
}
```


## 10. Data Structure Reference

The tables below show the public fields available on the main returned objects. The full API reference is in `docs/api.md`.


In [ ]:

display(Markdown("### Idea fields\n" + pc.schema_markdown(pc.Idea)))
display(Markdown("### WorkFeatures fields\n" + pc.schema_markdown(pc.WorkFeatures)))
display(Markdown("### EvidencePacket fields\n" + pc.schema_markdown(pc.EvidencePacket)))


### Idea fields
| Field | Type | Required |
| --- | --- | --- |
| id | <class 'str'> | required |
| title | <class 'str'> | required |
| thesis | <class 'str'> | required |
| mode | Literal['standard', 'calculus', 'scidialect_evo'] | required |
| novelty_claim | <class 'str'> | optional |
| mechanism_design | list[str] | optional |
| methodological_details | dict[str, Any] | optional |
| method_variants | list[str] | optional |
| why_it_might_work | list[str] | optional |
| validation_protocol | list[str] | optional |
| baselines | list[str] | optional |
| metrics | list[str] | optional |
| risks | list[str] | optional |
| assumptions | list[str] | optional |
| derived_principles | list[str] | optional |
| evidence_work_ids | list[str] | optional |
| source_evidence | list[dict[str, Any]] | optional |
| lineage | dict[str, Any] | optional |
| trace | dict[str, Any] | optional |
| generation_metadata | dict[str, Any] | optional |
| model | <class 'str'> | optional |
| run_id | <class 'str'> | optional |
| created_at | <class 'str'> | optional |

### WorkFeatures fields
| Field | Type | Required |
| --- | --- | --- |
| work_id | <class 'str'> | required |
| title | <class 'str'> | required |
| model | <class 'str'> | required |
| ideas | list[dict[str, Any]] | optional |
| principles | list[dict[str, Any]] | optional |
| baselines | list[dict[str, Any]] | optional |
| benchmarks | list[dict[str, Any]] | optional |
| takeaways | list[dict[str, Any]] | optional |
| result_facts | list[dict[str, Any]] | optional |
| source_excerpt_chars | <class 'int'> | optional |
| retained_pdf_path | <class 'str'> | optional |
| skipped | <class 'bool'> | optional |
| extraction_id | <class 'str'> | optional |
| created_at | <class 'str'> | optional |

### EvidencePacket fields
| Field | Type | Required |
| --- | --- | --- |
| query | <class 'str'> | optional |
| features | list[principia.models.WorkFeatures] | optional |
| user_note | <class 'str'> | optional |
| created_at | <class 'str'> | optional |

## 11. Compare Against Prior Ideas

This optional final step compares the generated idea with ideas extracted from the retrieved works.

The comparison progress display also uses heartbeat updates during long provider calls.


In [ ]:

compare_progress = pc.notebook_progress("Idea comparison")

comparison = ws.ideas.compare(
    idea,
    features,
    model=IDEA_MODEL,
    callback=compare_progress,
)

len(comparison.rows)


### Idea comparison

**Operation:** `ideas.compare`  
**Stage:** `complete`  
**Progress:** `####################` 100%  
**Elapsed:** 1m 12s  
**ETA:** 0s  
**Message:** Complete.  
**Counts:** llm_wait_seconds=70, rows=3

3

## 12. Review Comparison

In [ ]:

comparison_rows = [
    [
        row.get("title", ""),
        row.get("mechanistic_similarity", ""),
        row.get("essential_difference", ""),
        row.get("potential_advantage", ""),
    ]
    for row in comparison.rows[:8]
]

display(Markdown(pc.markdown_table(["Prior work", "Similarity", "Difference", "Advantage"], comparison_rows)))


| Prior work | Similarity | Difference | Advantage |
| --- | --- | --- | --- |
| Hierarchical Subsolar-Mass Mergers in Collapsar Disks | Both rely on disk fragmentation ($f_{disk}, \\Sigma_{frag}$) and repeated captures ($P_{capture}$) within a collapsar accretion disk to generate sub-solar fragments with non-zero orbital eccentricity ($e$) driven by dynamical kicks. | The prior work identifies eccentricity as a general dynamical outcome of hierarchical merging, whereas the generated idea quantifies a specific surviving eccentricity magnitude ($e \\approx 0.1$) due to the condition $t_{merge} < t_{circ}$ and explicitly links this value to time-variable non-thermal shock luminosity ($L_{nt} \\propto (1+e\\cos\\theta)^4$) and polarization signatures, rather than just kinematic properties. | Provides a concrete, observable electromagnetic discriminant (periodic spectral hardening and polarization variability) to distinguish the collapsar channel from the Primordial Black Hole (PBH) hypothesis, which predicts circular orbits ($e=0$) and no emission. |
| Superkilonova Formation Channel | Both posit that sub-solar mass objects form via fragmentation in a collapsar disk and merge shortly after a core-collapse supernova (Type IIb), utilizing the temporal association constraint ($< $ few days) between the SN explosion and GW merger. | The prior 'Superkilonova' model focuses on thermal emission from r-process nucleosynthesis (standard kilonova physics) which has been largely ruled out by non-detections; the generated idea shifts the mechanism to eccentricity-driven accretion shocks ($\\dot{M}(t)$ modulation) producing non-thermal radiation, bypassing the thermal constraints. | Revives the collapsar formation channel for S251112cm by predicting a spectral signature (non-thermal power law) that differs fundamentally from the thermal blackbody spectra excluded by current DECam/FTW limits. |
| Primordial Black Hole Interpretation of S251112cm | Both address the origin of the sub-solar mass candidate S251112cm and utilize the merger rate ($R$) and mass constraints to evaluate the viability of the proposed formation channel against observational data. | The PBH model assumes formation from early-Universe density fluctuations leading to vacuum mergers with circular orbits ($e \\to 0$) and no electromagnetic counterpart; the generated idea proposes an astrophysical disk environment where gas drag prevents full circularization ($e_{final} \\sim 0.1$) and generates EM emission via $L_{nt}$. | Offers a multi-messenger test (GW eccentricity + EM variability) that can definitively rule out the PBH dark matter hypothesis ($f_{PBH}$) if periodic non-thermal flaring is detected, whereas PBH models predict a null result in the EM domain. |

## 13. Local Outputs

Export the staged workflow result to local files.

The canonical export remains under hidden `.principia/artifacts/exports/`, and a visible mirror is written to:

```text
principia_astro/principia_outputs/latest/
  idea.md
  result.json
  works.json
```

The exported `idea.md` includes Methodological Details, formulas, validation plan, risks, source evidence, and comparison highlights.

The database counts show what is stored in SQLite. Rerun with `overwrite=True` only when you intentionally want to redo completed LLM extraction work.



In [ ]:
export_path = ws.export(
    goal=goal,
    works=works,
    features=features,
    idea=idea,
    comparison=comparison,
)

ws.counts(), export_path, ws.outputs_dir / "latest"


({'works': 122,
  'extractions': 6,
  'ideas': 1,
  'comparisons': 1,
  'runs': 12,
  'run_events': 1900},
 PosixPath('/home/supernova/Desktop/AI_projects/Principia/Principia-v1.3/examples/principia_astro/.principia/artifacts/exports/Eccentricity_Driven_Spectral_Signatures_In_Collapsar_Disk_Subsolar_Mergers'),
 PosixPath('/home/supernova/Desktop/AI_projects/Principia/Principia-v1.3/examples/principia_astro/principia_outputs/latest'))